**Importing the dataset**



In [18]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("rafsanmahmud333333/cifar-10-attacked-dataset")



Using Colab cache for faster access to the 'cifar-10-attacked-dataset' dataset.


**Data Preprocessing**

In [19]:
import os
import cv2
import pandas as pd
import numpy as np

# Set up paths to the dataset directory splits
base_path = os.path.join(path, "cifar10")
test_dir = os.path.join(base_path, "test")
clean_dir = os.path.join(test_dir, "clean")
fgsm_dir = os.path.join(test_dir, "fgsm")

# Load the test metadata to get the matching filenames and true labels
metadata_path = os.path.join(test_dir, "metadata_test(fgsm, bim, cANDw).csv")

if not os.path.exists(metadata_path):
    print(f"Error: Missing metadata file at {metadata_path}")
else:
    df_meta = pd.read_csv(metadata_path)

    # Filter metadata rows to select only the FGSM adversarial attack samples
    df_fgsm_meta = df_meta[df_meta['attack_type'] == 'fgsm']

    # Arrays to store the processed image matrices, noise residuals, and target labels
    clean_images = []
    fgsm_images = []
    noise_residuals = []
    labels = []

    print(f"Running data loading and image preprocessing for {len(df_fgsm_meta)} samples...")

    for index, row in df_fgsm_meta.iterrows():
        # Extract the raw filename string from the metadata path column
        full_path_str = row['image_path']
        filename = os.path.basename(full_path_str)
        true_label = row['original_label']

        clean_img_path = os.path.join(clean_dir, filename)
        fgsm_img_path = os.path.join(fgsm_dir, filename)

        if os.path.exists(clean_img_path) and os.path.exists(fgsm_img_path):
            # Load images as 2D coordinate matrices (uint8 arrays in BGR format)
            bgr_clean = cv2.imread(clean_img_path)
            bgr_fgsm = cv2.imread(fgsm_img_path)

            # Enforce identical M x N spatial dimensions (32x32)
            if bgr_clean.shape[:2] != (32, 32):
                bgr_clean = cv2.resize(bgr_clean, (32, 32), interpolation=cv2.INTER_AREA)
            if bgr_fgsm.shape[:2] != (32, 32):
                bgr_fgsm = cv2.resize(bgr_fgsm, (32, 32), interpolation=cv2.INTER_AREA)

            # Convert color space from BGR channel layout to standard RGB
            rgb_clean = cv2.cvtColor(bgr_clean, cv2.COLOR_BGR2RGB)
            rgb_fgsm = cv2.cvtColor(bgr_fgsm, cv2.COLOR_BGR2RGB)

            # Normalize pixel intensity values from [0, 255] to [0.0, 1.0] float space
            norm_clean = rgb_clean.astype(np.float32) / 255.0
            norm_fgsm = rgb_fgsm.astype(np.float32) / 255.0

            # Perform element-wise subtraction to isolate the adversarial noise matrix η(x,y)
            noise_matrix = norm_fgsm - norm_clean

            # Append matrices and labels to their respective lists
            clean_images.append(norm_clean)
            fgsm_images.append(norm_fgsm)
            noise_residuals.append(noise_matrix)
            labels.append(true_label)

    # Convert the collected image structures into final NumPy tensors
    X_clean = np.array(clean_images)
    X_fgsm = np.array(fgsm_images)
    X_noise = np.array(noise_residuals)
    y_test = np.array(labels)

    # Verification summary to check dataset tensor shapes
    print("\n--- Preprocessing Pipeline Verification ---")
    print(f"X_clean Matrix Shape : {X_clean.shape}")
    print(f"X_fgsm Matrix Shape  : {X_fgsm.shape}")
    print(f"X_noise Matrix Shape : {X_noise.shape}")
    print(f"y_test Target Shape  : {y_test.shape}")

Running data loading and image preprocessing for 500 samples...

--- Preprocessing Pipeline Verification ---
X_clean Matrix Shape : (500, 32, 32, 3)
X_fgsm Matrix Shape  : (500, 32, 32, 3)
X_noise Matrix Shape : (500, 32, 32, 3)
y_test Target Shape  : (500,)
